# grid

> Gridable detection and paged array slicing

In [ ]:
#| default_exp grid

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
import numpy as np, pandas as pd
from paar.grid import (is_gridable, array_page, array_detail, frame_detail, series_detail,
                       ArrayResolver, FrameResolver, SeriesResolver, _MAX_ELEMS)
from paar.resolvers import resolve_for

def test_is_gridable():
    assert is_gridable(np.arange(6).reshape(2,3))
    assert is_gridable(np.arange(3))
    assert is_gridable(pd.DataFrame({'a':[1,2]}))
    assert is_gridable(pd.Series([1,2,3]))
    assert not is_gridable(np.zeros((2,2,2)))   # 3-D not gridable
    assert not is_gridable([1,2,3]) and not is_gridable(5)
def test_page_ndarray_2d():
    p = array_page(np.arange(6).reshape(2,3))
    assert p['nrows']==2 and p['ncols']==3
    assert p['headers']==['0','1','2'] and p['index']==['0','1']
    assert p['cells']==[['0','1','2'],['3','4','5']]
def test_page_ndarray_1d():
    p = array_page(np.arange(3))
    assert p['headers']==['0'] and p['index']==['0','1','2'] and p['cells']==[['0'],['1'],['2']]
    assert p['nrows']==3 and p['ncols']==1
def test_page_dataframe():
    p = array_page(pd.DataFrame({'x':[1,2],'y':[3,4]}))
    assert p['headers']==['x','y'] and p['index']==['0','1']
    assert p['cells']==[['1','3'],['2','4']] and p['nrows']==2 and p['ncols']==2
def test_page_series():
    p = array_page(pd.Series([10,20], name='s'))
    assert p['headers']==['s'] and p['cells']==[['10'],['20']] and p['nrows']==2 and p['ncols']==1
def test_page_paging_offsets():
    p = array_page(np.arange(10).reshape(10,1), roff=5, rows=3)
    assert p['index']==['5','6','7'] and p['cells']==[['5'],['6'],['7']] and p['nrows']==10 and p['roff']==5
    assert p['rows']==3
def test_page_non_gridable_none():
    assert array_page([1,2,3]) is None
def test_array_detail_stats_then_elements():
    v = np.arange(6).reshape(2,3)
    names = [nm for nm,_,_ in array_detail(v)]
    assert names[:3]==['shape','dtype','size']
    assert {'min','max','mean','std','sum'} <= set(names)
    assert names[-2:]==['0','1']   # two rows as element children
def test_array_detail_skips_stats_on_non_numeric():
    names = [nm for nm,_,_ in array_detail(np.array(['a','b','c']))]
    assert 'mean' not in names and 'sum' not in names   # numeric reductions skipped
    assert names[:2]==['shape','dtype'] and names[-3:]==['0','1','2']
def test_frame_detail_columns_as_children():
    names = [nm for nm,_,_ in frame_detail(pd.DataFrame({'x':[1,2],'y':[3,4]}))]
    assert names[:3]==['shape','size','dtypes'] and names[-2:]==['x','y']
def test_series_detail_elements():
    d = series_detail(pd.Series([10,20], name='s'))
    names = [nm for nm,_,_ in d]
    assert names[:3]==['dtype','size','name']
    assert d[-2][2]==10 and d[-1][2]==20   # element values
def test_gridable_resolvers_registered():
    assert isinstance(resolve_for(np.arange(3)), ArrayResolver)
    assert isinstance(resolve_for(pd.DataFrame({'a':[1]})), FrameResolver)
    assert isinstance(resolve_for(pd.Series([1])), SeriesResolver)
def test_detail_caps_large():
    big = array_detail(np.arange(10000))
    assert len([r for r in big if r[1].startswith('[')]) == _MAX_ELEMS   # only 90 element rows, not 10000
    assert big[-1][0]=='…' and 'more' in big[-1][2]
    assert not any(r[0]=='…' for r in array_detail(np.arange(3)))        # small: no sentinel
    srows = series_detail(pd.Series(range(5000)))
    assert srows[-1][0]=='…'
for t in [test_is_gridable,test_page_ndarray_2d,test_page_ndarray_1d,test_page_dataframe,
          test_page_series,test_page_paging_offsets,test_page_non_gridable_none,
          test_array_detail_stats_then_elements,test_array_detail_skips_stats_on_non_numeric,
          test_frame_detail_columns_as_children,test_series_detail_elements,
          test_gridable_resolvers_registered,test_detail_caps_large]: t()

In [ ]:
#| export
import itertools
from paar.reprs import safe_repr
from paar.resolvers import Resolver, register_resolver
try: import numpy as np
except Exception: np = None
try: import pandas as pd
except Exception: pd = None

def is_gridable(v):
    "True for a numpy ndarray (1-2D) or a pandas DataFrame/Series."
    if np is not None and isinstance(v, np.ndarray): return v.ndim in (1, 2)
    if pd is not None and isinstance(v, (pd.DataFrame, pd.Series)): return True
    return False

def _cell(x): return safe_repr(x, 40)

def array_page(v, roff=0, coff=0, rows=50, cols=50):
    "Page a gridable value -> dict(headers,index,cells,nrows,ncols,roff,coff,rows,cols), or None."
    if pd is not None and isinstance(v, pd.DataFrame):
        nrows, ncols = v.shape
        sub = v.iloc[roff:roff+rows, coff:coff+cols]
        headers = [str(c) for c in sub.columns]
        index = [str(i) for i in sub.index]
        cells = [[_cell(x) for x in row] for row in sub.values.tolist()]
    elif pd is not None and isinstance(v, pd.Series):
        nrows, ncols = v.shape[0], 1
        sub = v.iloc[roff:roff+rows]
        headers = [str(v.name) if v.name is not None else 'value']
        index = [str(i) for i in sub.index]
        cells = [[_cell(x)] for x in sub.tolist()]
    elif np is not None and isinstance(v, np.ndarray) and v.ndim == 2:
        nrows, ncols = v.shape
        sub = v[roff:roff+rows, coff:coff+cols]
        headers = [str(c) for c in range(coff, min(coff+cols, ncols))]
        index = [str(r) for r in range(roff, min(roff+rows, nrows))]
        cells = [[_cell(x) for x in row] for row in sub.tolist()]
    elif np is not None and isinstance(v, np.ndarray) and v.ndim == 1:
        nrows, ncols = v.shape[0], 1
        sub = v[roff:roff+rows]
        headers = ['0']
        index = [str(r) for r in range(roff, min(roff+rows, nrows))]
        cells = [[_cell(x)] for x in sub.tolist()]
    else:
        return None
    return {'headers': headers, 'index': index, 'cells': cells,
            'nrows': nrows, 'ncols': ncols, 'roff': roff, 'coff': coff,
            'rows': rows, 'cols': cols}

In [ ]:
#| export
_MAX_ELEMS = 90   # element rows generated per expand (metadata rows add on top; keeps total under snapshot MAX_CHILDREN)

def _stat(fn):
    "Call a numpy/pandas reduction, returning None if it raises (e.g. non-numeric data)."
    try: return fn()
    except Exception: return None

def array_detail(v):
    "Curated child rows for an ndarray: metadata, numeric stats, then indexed elements (capped)."
    rows = [('shape', '.shape', v.shape), ('dtype', '.dtype', v.dtype), ('size', '.size', v.size)]
    rows += [(nm, f'.{nm}()', s) for nm,fn in [('min',v.min),('max',v.max),('mean',v.mean),
                                               ('std',v.std),('sum',v.sum)] if (s:=_stat(fn)) is not None]
    rows += [(str(i), f'[{i}]', x) for i,x in itertools.islice(enumerate(v), _MAX_ELEMS)]
    n = len(v)
    if n > _MAX_ELEMS: rows.append(('…', '', f'{n-_MAX_ELEMS} more'))
    return rows

def frame_detail(v):
    "Curated child rows for a DataFrame: metadata then each column as a Series."
    return [('shape', '.shape', v.shape), ('size', '.size', v.size), ('dtypes', '.dtypes', v.dtypes),
            *[(str(c), f'[{c!r}]', v[c]) for c in v.columns]]

def series_detail(v):
    "Curated child rows for a Series: metadata then indexed elements (capped)."
    rows = [('dtype', '.dtype', v.dtype), ('size', '.size', v.size), ('name', '.name', v.name)]
    rows += [(str(ix), f'.iloc[{i}]', x) for i,(ix,x) in itertools.islice(enumerate(v.items()), _MAX_ELEMS)]
    n = len(v)
    if n > _MAX_ELEMS: rows.append(('…', '', f'{n-_MAX_ELEMS} more'))
    return rows

class ArrayResolver(Resolver):
    "Expands an ndarray into stats + element rows (peers the grid table)."
    def can(self, v): return np is not None and isinstance(v, np.ndarray) and v.ndim >= 1
    def children(self, v): return array_detail(v)

class FrameResolver(Resolver):
    "Expands a DataFrame into metadata + column Series."
    def can(self, v): return pd is not None and isinstance(v, pd.DataFrame)
    def children(self, v): return frame_detail(v)

class SeriesResolver(Resolver):
    "Expands a Series into metadata + element rows."
    def can(self, v): return pd is not None and isinstance(v, pd.Series)
    def children(self, v): return series_detail(v)

for _r in (ArrayResolver(), FrameResolver(), SeriesResolver()): register_resolver(_r)

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()